In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader
from torch import nn, optim
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from evaluate import load as load_metric
from torch.utils.tensorboard import SummaryWriter
import os

from models.mambaPlusPlus import MambaPlusPlusML
from models.transformer import Transformer

# === Настройка ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8
num_epochs = 3
max_length = 128

# === Токенизатор ===
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
vocab_size = len(tokenizer)

# === Модельные параметры ===
embed_dim = 512
heads = 4

def create_models():
    mamba = MambaPlusPlusML(vocab_size=vocab_size, embed_dim=embed_dim, hidden_dim=embed_dim,
                             num_heads=heads, num_layers=1).to(device)
    transformer = Transformer(vocab_size, embed_dim, nhead=heads, num_layers=1).to(device)
    return mamba, transformer

# === Токенизация ===
def tokenize_fn(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=max_length)

# === Collate ===
def collate_fn(batch):
    input_ids = [torch.tensor(example["input_ids"]) for example in batch]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    return {"input_ids": input_ids}

# === Train ===
def train_model(model, dataloader, writer, tag, epochs=1):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    counter = 0
    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(dataloader, desc=f"Training {tag} Epoch {epoch+1}"):
            counter += 1
            input_ids = batch["input_ids"].to(device)
            labels = input_ids.clone()

            optimizer.zero_grad()
            logits = model(input_ids)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            avg_loss = total_loss / 8 / counter
            writer.add_scalar(f"TrainLoss", avg_loss, counter)
        print(f"[{tag}] Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")

# === Evaluate ===
def evaluate_model(model, dataloader, tag):
    metric = load_metric("accuracy")
    model.eval()
    total_acc = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {tag}"):
            input_ids = batch["input_ids"].to(device)
            labels = input_ids.clone()
            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)
            metric.add_batch(predictions=preds[:, -1].cpu().numpy(), references=labels[:, -1].cpu().numpy())

    result = metric.compute()
    return result

# === Основной запуск экспериментов ===
def run_experiment():
    datasets_info = {
        "LAMBADA": ("cimec/lambada", "text", None),
        "MMLU": ("cais/mmlu", "input", "target"),
    }

    for name, (path, input_col, target_col) in datasets_info.items():
        print(f"\n📚 Dataset: {name}")

        # === Загрузка ===
        dataset = load_dataset(path, split="train")
        dataset = dataset.filter(lambda x: x.get(input_col) is not None)

        if target_col:
            dataset = dataset.filter(lambda x: x.get(target_col) is not None)
            dataset = dataset.map(lambda x: {"text": x[input_col] + " " + x[target_col]})
        else:
            dataset = dataset.map(lambda x: {"text": x[input_col]})

        dataset = dataset.train_test_split(test_size=0.2, seed=42)
        train_set = dataset["train"].map(tokenize_fn, batched=True).remove_columns(["text"])
        test_set = dataset["test"].map(tokenize_fn, batched=True).remove_columns(["text"])

        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
        test_loader = DataLoader(test_set, batch_size=batch_size, collate_fn=collate_fn)

        # === Обучение и оценка Mamba++
        mamba_model, transformer_model = create_models()
        print("🔧 Training Mamba++...")
        writer = SummaryWriter(log_dir="/home/adanilishin/mambaProject/tensorboard/{}/{}".format(name, "Mamba++"))
        train_model(mamba_model, train_loader, writer, f"Mamba++", epochs=num_epochs)
        print("📊 Evaluating Mamba++...")
        score = evaluate_model(mamba_model, test_loader, f"Mamba++")
        print(f"Accuracy: {score}")

        # === Обучение и оценка Transformer
        print("🔧 Training Transformer...")
        writer = SummaryWriter(log_dir="/home/adanilishin/mambaProject/tensorboard/{}/{}".format(name, "Transformer"))
        train_model(transformer_model, train_loader, writer, f"Transformer", epochs=num_epochs)
        print("📊 Evaluating Transformer...")
        score = evaluate_model(transformer_model, test_loader, f"Transformer")
        print(f"Accuracy: {score}")
        writer.close()

run_experiment()


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader
from torch import nn, optim
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from evaluate import load as load_metric
from torch.utils.tensorboard import SummaryWriter

from models.mambaPlusPlus import MambaPlusPlusML
from models.transformer import Transformer

# === Настройка ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 2
num_epochs = 3
max_length = 128

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
vocab_size = len(tokenizer)

embed_dim = 512
heads = 4

mamba_model = MambaPlusPlusML(
    vocab_size=vocab_size, embed_dim=embed_dim, hidden_dim=embed_dim,
    num_heads=heads, num_layers=1
).to(device)

transformer_model = Transformer(
    vocab_size, embed_dim, nhead=heads, num_layers=1
).to(device)

def tokenize_fn(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=max_length)

def evaluate_model(model, dataloader):
    metric = load_metric("accuracy")
    model.eval()
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            labels = input_ids.clone()
            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)
            metric.add_batch(predictions=preds[:, -1].cpu().numpy(), references=labels[:, -1].cpu().numpy())
    return metric.compute()

def train_model(model, dataloader, writer, tag, epochs=1):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(dataloader, desc=f"Training Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            labels = input_ids.clone()
            optimizer.zero_grad()
            logits = model(input_ids)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} - {tag} Avg Loss: {avg_loss:.4f}")
        writer.add_scalar(f"{tag}/loss", avg_loss, epoch)

def collate_fn(batch):
    input_ids = [torch.tensor(example["input_ids"]) for example in batch]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    return {"input_ids": input_ids}

def run_experiment():
    datasets_info = {
        "LAMBADA": ("cimec/lambada", "text", None),
        "MMLU": ("cais/mmlu", "input", "target"),
    }

    for name, (path, input_col, target_col) in datasets_info.items():
        print(f"\n📚 Dataset: {name}")
        writer = SummaryWriter(log_dir=f"runs/{name}")

        # === Загрузка ===
        if "mmlu" in path:
            dataset = load_dataset(path, name="all", split="test")
        else:
            dataset = load_dataset(path, split="train")

        dataset = dataset.filter(lambda x: x.get(input_col) is not None)
        if target_col:
            dataset = dataset.filter(lambda x: x.get(target_col) is not None)
            dataset = dataset.map(lambda x: {"text": x[input_col] + " " + x[target_col]})
        else:
            dataset = dataset.map(lambda x: {"text": x[input_col]})
        dataset = dataset.remove_columns([col for col in dataset.column_names if col != "text"])
        dataset = dataset.map(tokenize_fn, batched=True)

        # === Обучение и тест ===
        if name == "LAMBADA":
            dataset = dataset.train_test_split(test_size=0.2, seed=42)
            train_set = dataset["train"]
            test_set = dataset["test"]

            train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
            test_loader = DataLoader(test_set, batch_size=batch_size, collate_fn=collate_fn)

            print("🔧 Training Mamba++...")
            train_model(mamba_model, train_loader, writer, tag=f"{name}/Mamba++", epochs=num_epochs)

            print("📊 Evaluating Mamba++...")
            score = evaluate_model(mamba_model, test_loader)
            print(f"Accuracy: {score}")
            writer.add_scalar(f"{name}/Mamba++/accuracy", score["accuracy"], 0)

            print("🔧 Training Transformer...")
            train_model(transformer_model, train_loader, writer, tag=f"{name}/Transformer", epochs=num_epochs)

            print("📊 Evaluating Transformer...")
            score = evaluate_model(transformer_model, test_loader)
            print(f"Accuracy: {score}")
            writer.add_scalar(f"{name}/Transformer/accuracy", score["accuracy"], 0)

        elif name == "MMLU":
            test_loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)

            print("📊 Evaluating Mamba++...")
            score = evaluate_model(mamba_model, test_loader)
            print(f"Accuracy: {score}")
            writer.add_scalar(f"{name}/Mamba++/accuracy", score["accuracy"], 0)

            print("📊 Evaluating Transformer...")
            score = evaluate_model(transformer_model, test_loader)
            print(f"Accuracy: {score}")
            writer.add_scalar(f"{name}/Transformer/accuracy", score["accuracy"], 0)

        writer.close()

run_experiment()


/home/adanilishin/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/adanilishin/miniconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(



📚 Dataset: LAMBADA


Map:  38%|███▊      | 1000/2662 [01:33<02:06, 13.18 examples/s]

: 